In [1]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-23.jdk/Contents/Home"  # or wherever your JDK 17/21 lives
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder.appName("Local Spark Session")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "spark-warehouse")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .enableHiveSupport()
    .getOrCreate()
)
print(spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/10 07:32:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://127.0.0.1:4040


# Chapter 4 - Structured API Overview

These APIs refer to three core types of distributed collection APIs:
* Datasets;
* DataFrames;
* SQL Tables and Views;

The majority of the Structured APIs apply to both, batch and streaming computation. This means, it should be easy to switch from one to the other and vice versa.

## Review
```
Spark is a distributed programming model in which the user specifies transformations. Multiple transformations build up a directed acyclic graph of instructions. An action begins the process of executing the graph of instructions, as a single job, by breaking it down into stages and tasks to execute across the cluster.
```


**Catalyst** - Is an engine that is executed inside spark that maintains its own type information through the planning and processing of work. Spark data types map directly to the different language APIs that spark maintains.

## Structured APIS
Spark Structured APIs have two main types:
* DataFrames (untyped): Spark checks the data types **at runtime** against the schema. They are the most commonly used API;
* Datasets (typed): Spark checks types **at compile time**, providing stronger type safety. They are only available in Scala and Java.

In Scala, a DataFrame is essentially a Dataset[Row]. Row is Spark's internal representation of data, optimized for efficient computation.
This internal format is faster and more memory-efficient than using normal JVM objects because it reduces **object creation and garbage collection overhead**.

## Spark Query execution
When you write a DataFrame, Dataset or SQL code, Spark goes through these steps:
1. Write the code -> You create a DataFrame/Dataset/SQL Query
2. Logical Plan -> Spark converts your code into a Logical Plan describing what you want to do, without deciding exactly how.
3. Physical Plan -> The Catalyst Optimizer analyzes and optimizes the Logical Plan and decides how to execute it.
4. Execute on the cluster -> Spark Converts the physical Plan into operations (RDD manipulations) and distributes them across the cluster
5. Return the result -> The cluster executes the work and sends the result back.

![alt text](image.png)

## Logical Plan
![alt text](image-1.png)

The logical Plan describes what you want Spark to do, without worrying about executors, drivers, or how the work will physically run.
The process is:
1. User code -> Unresolved Logical Plan
   1. Spark converts your DataFrame/SQL code into a logical plan;
   2. It is unresolved because Spark hasn't yet verified that the referenced tables and columns actually exist;
2. Analyzer
   1. Spark checks the catalog, which contains information about the tables, columns and DataFrames.
   2. If a table or column doesn't exist -> The query is rejected;
   3. If everything exists -> The plan becomes resolved;
3. Catalyst Optimizer:
   1. Spark optimizes the resolved logical plan using optimization rules;
   2. For example, it can push **filters/predicates closer to the data source**, reducing the amount of data that needs to be processed.
4. Extensibility:
   1. Spark allows external packages to add their own Catalyst Optimization rules for specialized use cases.

## Physical Planning
![alt text](image-2.png)

Once Spark has an optimized Logical Plan, it creates the Physical Plan, which determines how the query will actually run on the cluster.
1. Generate execution strategies: Spark considers different ways to execute the query;
2. Cost-based comparison: Spark compares these strategies using a cost model;
   1. Example: Deciding which join strategy to use based on the table/partition size;
3. Choose the best strategy: Spark selects the most efficient physical execution plan;
4. Generate RDD transformations: The physical plan is ultimately translated into RDDs and transformations that run across the cluster.

## Execution
Upon selecting a physical plan, Spark runs all of this code over RDDs, the lower-level programming interface of Spark. Spark performs further optimizations at runtime, generating native Java bytecode that can remove entire tasks or stages during execution.